# Stage 03: Localised (window-level) PCL detection (Part 4)

Utilises task2 dataset span text to get pure localised signals of where in paragh pcl is actually occuring, so the model can aggreagate these local signals and global context for more accurate paragraph-level predictions

Approach: Using span level PCL info to get window level PCL info (windows are LQ, median & UQ of span length - aligned with span shape)
1. labels each token if they are present in positive span or not
2. For each window gets the window average of labels, apply overlap threshold to get signal (smooths span boundaries & encourages full use of span for strong signal)
3. Get window average of token logits at final layer, BCE on corresponding window label at that token for every token (at a stride), average across all windows: window_loss
4. Max logit across all windows is then put into BCE and then gives: paragraph loss
5. total_loss = paragraph_loss  +  lambda * window_loss

## Imports & Dataset utilities

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
import os
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers import get_linear_schedule_with_warmup

# project root (one level above notebooks)
ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

from src.data.make_dataset import build_task1_task2_with_spans, validate_span_ranges, validate_span_text_alignment, truncation_rate_by_label
from src.training.metrics import stats_on_loader
from src.training.tokenization_utils import ensure_token_cache, TensorCacheDataset, PCLTokenDataset

RAW_TASK1 = ROOT / "data" / "raw" / "dontpatronizeme_pcl.tsv"
RAW_TASK2 = ROOT / "data" / "raw" / "dontpatronizeme_categories.tsv"
TRAIN_SPLIT = ROOT / "data" / "splits" / "train_semeval_parids-labels.csv"
DEV_SPLIT = ROOT / "data" / "splits" / "dev_semeval_parids-labels.csv"

train_df, dev_df, pcl_df, spans_df_norm = build_task1_task2_with_spans(
    raw_task1_path=RAW_TASK1,
    raw_task2_path=RAW_TASK2,
    train_split_path=TRAIN_SPLIT,
    dev_split_path=DEV_SPLIT,
)

print("train_df:", train_df.shape, "| positives:", int(train_df["label_bin"].sum()))
print("dev_df:", dev_df.shape, "| positives:", int(dev_df["label_bin"].sum()))
print("Example span_ranges:", train_df.loc[train_df["label_bin"].idxmax(), "span_ranges"] if len(train_df) else None)

/home/joshua_killa/.pyenv/versions/pcl-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train_df: (8375, 8) | positives: 794
dev_df: (2094, 8) | positives: 199
Example span_ranges: [(157, 242)]


In [2]:
# ---- choose backbone here ----
MODEL_CANDIDATES = {
    "deberta": "microsoft/deberta-v3-base",
    "albert": "albert-base-v2",
    "albert_large": "albert-large-v2",
}

MODEL_KEY = os.getenv("PCL_BACKBONE", "albert_large")  # set to "albert" to try ALBERT
MODEL_NAME = MODEL_CANDIDATES[MODEL_KEY]
MAX_LEN = 192
WINDOW_SIZES = [8, 14, 23]

def load_tokenizer(model_name: str):
    # Online first, then local cache fallback (for DNS/no-internet issues)
    try:
        return AutoTokenizer.from_pretrained(model_name, use_fast=True)
    except Exception as e:
        print(f"Tokenizer load failed ({type(e).__name__}: {e}). Trying local cache...")
        return AutoTokenizer.from_pretrained(model_name, use_fast=True, local_files_only=True)

tokenizer = load_tokenizer(MODEL_NAME)



In [3]:
def window_mean_1d(x, window_size: int, stride: int = 1):
    """
    x: (B, T)
    returns: (B, Nw) where Nw = floor((T-window_size)/stride)+1
    """
    B, T = x.shape
    if T < window_size:
        return x.new_zeros((B, 0))
    windows = x.unfold(dimension=1, size=window_size, step=stride)  # (B, Nw, window)
    return windows.mean(dim=-1)  # (B, Nw)

def window_overlap_labels(token_labels, window_size: int, tau: float = 0.2, stride: int = 1):
    """
    token_labels: (B, T) in {0,1}
    returns: (B, Nw) in {0,1} based on mean>=tau
    """
    frac = window_mean_1d(token_labels.float(), window_size, stride)  # overlap fraction
    return (frac >= tau).float()

def mask_token_logits(token_logits, attention_mask):
    """
    token_logits: (B,T)
    attention_mask: (B,T) 1=real, 0=pad
 
    For paragraph max pooling, pads should be very negative (dtype-safe).
    For window means, pads should be 0.
    """
    # dtype-safe "very negative"
    neg_inf = torch.tensor(-1e4, device=token_logits.device, dtype=token_logits.dtype)
 
    token_logits_maxsafe = token_logits.masked_fill(attention_mask == 0, neg_inf)
    token_logits_meansafe = token_logits.masked_fill(attention_mask == 0, 0.0)
    return token_logits_meansafe, token_logits_maxsafe

In [4]:
# Window pooling for token logits -> paragraph logit + window logits
# (replaces LogitPooler)

class WindowLogitPooler(nn.Module):
    """
    token_logits: (B,T)
    attention_mask: (B,T) bool
    returns:
      - paragraph_logit: (B,)
      - windows: list of (w, win_logits(B,Nw), win_mask(B,Nw))
    """
    def __init__(self, window_sizes, stride=1, sentinel=-1e9):
        super().__init__()
        self.window_sizes = [int(w) for w in window_sizes]
        self.stride = int(stride)

    def forward(self, token_logits: torch.Tensor, attention_mask: torch.Tensor):
        # mask token logits for max pooling safety + mean pooling safety
        token_logits_meansafe, token_logits_maxsafe = mask_token_logits(token_logits, attention_mask)

        para_logits_candidates = []
        windows = []

        for w in self.window_sizes:
            win_logits = window_mean_1d(token_logits_meansafe, w, stride=self.stride)  # (B,Nw)

            # Build a window mask: window is valid if it contains at least one real token.
            # We do it by unfolding the attention mask and checking any==1
            if attention_mask.shape[1] >= w:
                win_mask = attention_mask.to(torch.float32).unfold(1, w, self.stride).sum(dim=-1) > 0  # (B,Nw) bool
            else:
                win_mask = token_logits.new_zeros((token_logits.size(0), 0), dtype=torch.bool)

            windows.append((w, win_logits, win_mask))

            if win_logits.shape[1] > 0:
                # masked max over windows
                neg_inf = torch.finfo(win_logits.dtype).min
                win_logits_safe = win_logits.masked_fill(~win_mask, neg_inf)
                para_logits_candidates.append(win_logits_safe.max(dim=1).values)  # (B,)
            else:
                # fallback: token-level max
                para_logits_candidates.append(token_logits_maxsafe.max(dim=1).values)

        # paragraph logit = max across window sizes
        paragraph_logit = torch.stack(para_logits_candidates, dim=1).max(dim=1).values  # (B,)

        return paragraph_logit, windows

In [5]:
class TokenCLSModel(nn.Module):
    def __init__(
        self,
        model_name,
        window_sizes,
        stride=1,
        tau=0.2,
        alpha_window=1.0,
        normalize_window_loss=True,
    ):
        super().__init__()

        try:
            self.encoder = AutoModel.from_pretrained(model_name)
        except Exception as e:
            print(f"Model load failed ({type(e).__name__}: {e}). Trying local cache...")
            self.encoder = AutoModel.from_pretrained(model_name, local_files_only=True)

        hidden = self.encoder.config.hidden_size
        self.token_head = nn.Linear(hidden, 1)

        self.tau = float(tau)
        self.alpha_window = float(alpha_window)
        self.normalize_window_loss = bool(normalize_window_loss)

        self.pooler = WindowLogitPooler(window_sizes=window_sizes, stride=stride)

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
        token_labels=None,
        token_loss_mask=None,     # kept for compatibility, but not needed
        paragraph_label=None,
    ):
        # Forward encoder
        try:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        except TypeError:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        hidden = outputs.last_hidden_state  # (B,T,H)
        hidden = hidden.to(dtype=self.token_head.weight.dtype)

        token_logits = self.token_head(hidden).squeeze(-1)  # (B,T)

        # Paragraph logit from windows
        paragraph_logit, windows = self.pooler(token_logits, attention_mask.bool())

        # Paragraph loss
        loss_par = None
        if paragraph_label is not None:
            loss_par = F.binary_cross_entropy_with_logits(
                paragraph_logit.float(),
                paragraph_label.float(),
            )

        # Window loss (span supervision)
        loss_win = None
        if token_labels is not None:
            win_losses = []
            for (w, win_logits, win_mask) in windows:
                if win_logits.shape[1] == 0:
                    continue

                win_labels = window_overlap_labels(token_labels, w, tau=self.tau, stride=self.pooler.stride)  # (B,Nw)

                # Only compute loss on valid windows
                # Mask both logits and labels to valid entries, then BCE
                # (flatten masked entries)
                m = win_mask
                if m.any():
                    l = F.binary_cross_entropy_with_logits(
                        win_logits[m].float(),
                        win_labels[m].float(),
                    )
                    win_losses.append(l)

            if len(win_losses) > 0:
                loss_win = torch.stack(win_losses).mean()
                if self.normalize_window_loss:
                    # already mean over window-sizes; this flag kept for future tweaks
                    loss_win = loss_win

        # Total loss (ONE relative weight)
        loss = None
        if loss_par is not None and loss_win is not None:
            loss = loss_par + self.alpha_window * loss_win
        elif loss_par is not None:
            loss = loss_par
        elif loss_win is not None:
            loss = self.alpha_window * loss_win

        return {
            "loss": loss,
            "loss_par": loss_par,
            "loss_tok": loss_win,           # keep key name so your logging doesn't break
            "paragraph_logit": paragraph_logit,
            "token_logits": token_logits,
        }

In [6]:
CACHE_DIR = ROOT / "data" / "cache" / f"pcl_tok_{MODEL_KEY}_len{MAX_LEN}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_cache = ensure_token_cache(
    train_df,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    cache_path=CACHE_DIR / "train.pt",
    model_key=MODEL_KEY,
    model_name=MODEL_NAME,
)
dev_cache = ensure_token_cache(
    dev_df,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    cache_path=CACHE_DIR / "dev.pt",
    model_key=MODEL_KEY,
    model_name=MODEL_NAME,
)

train_dataset = TensorCacheDataset(train_cache)
dev_dataset = TensorCacheDataset(dev_cache)

print("train_dataset:", type(train_dataset), "len=", len(train_dataset))
print("dev_dataset:", type(dev_dataset), "len=", len(dev_dataset))

train_dataset: <class 'src.training.tokenization_utils.TensorCacheDataset'> len= 8375
dev_dataset: <class 'src.training.tokenization_utils.TensorCacheDataset'> len= 2094


In [7]:
from torch.optim import AdamW
from tqdm import tqdm
import torch
import numpy as np
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _amp_settings(model_key: str, model_name: str):
    key = (model_key or "").lower()
    name = (model_name or "").lower()

    if ("albert" in key) or ("albert" in name):
        enabled = torch.cuda.is_available()
        return enabled, torch.float16, True  # FP16 + GradScaler

    if ("deberta" in key) or ("deberta" in name):
        enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        return enabled, torch.bfloat16, False  # BF16, no GradScaler

    return False, None, False

USE_AMP, AMP_DTYPE, NEEDS_SCALER = _amp_settings(MODEL_KEY, MODEL_NAME)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))

print(
    f"DEVICE={DEVICE} | backbone={MODEL_KEY} | amp={USE_AMP} | amp_dtype={AMP_DTYPE} | "
    f"scaler={bool(scaler.is_enabled())}"
)

model = TokenCLSModel(
    MODEL_NAME,
    window_sizes=WINDOW_SIZES,
    stride=1,
    tau=0.2,
    alpha_window=1.0,
).to(DEVICE)

# train_dataset = PCLTokenDataset(train_df)

# ---- balanced paragraph sampling (approx 50/50 pos/neg) ----
labels = train_df["label_bin"].astype(int).to_numpy()
pos = int(labels.sum())
neg = int(len(labels) - pos)

w_pos = 1.0 / max(pos, 1)
w_neg = 1.0 / max(neg, 1)

sample_weights = torch.tensor([w_pos if y == 1 else w_neg for y in labels], dtype=torch.double)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),  # keep "epoch" size comparable to dataset size
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=sampler,   # <- sampler replaces shuffle
)

# separate loaders for F1 computation (no shuffle)
train_eval_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
# dev_dataset = PCLTokenDataset(dev_df)
dev_eval_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

optimizer = AdamW(
    model.parameters(),
    lr=7e-6,
    weight_decay=1e-2,
)

THRESH = 0.5
GRAD_ACCUM = 2  # <- match your optuna best configs; effective batch = 16 * 2 = 32
EPOCHS = 7

total_update_steps = (len(train_loader) * EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
warmup_steps = int(0.07 * total_update_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)



DEVICE=cuda | backbone=albert_large | amp=True | amp_dtype=torch.float16 | scaler=True


/tmp/ipykernel_440919/3102015375.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 503.16it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# # Optional curriculum: token-only warm start
# TOKEN_ONLY_EPOCHS = 1
# for p in model.paragraph_head.parameters():
#     p.requires_grad = True  # default

# for epoch in range(EPOCHS):
#     token_only = epoch < TOKEN_ONLY_EPOCHS

#     # ---- curriculum toggle ----
#     if token_only:
#         for p in model.paragraph_head.parameters():
#             p.requires_grad = False
#         model.lambda_token = 1.0
#     else:
#         for p in model.paragraph_head.parameters():
#             p.requires_grad = True
#         model.lambda_token = 0.3

#     model.train()
#     total_loss = 0.0
#     optimizer.zero_grad(set_to_none=True)

#     last_step = 0
#     for step, batch in enumerate(tqdm(train_loader), start=1):
#         last_step = step
#         batch = {k: v.to(DEVICE) for k, v in batch.items()}

#         # TRUE token-only warm start: disable paragraph loss entirely
#         if token_only:
#             batch["paragraph_label"] = None

#         if USE_AMP:
#             with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
#                 out = model(**batch)
#                 loss = out["loss"]
#         else:
#             out = model(**batch)
#             loss = out["loss"]

#         if torch.isnan(loss) or torch.isinf(loss):
#             raise RuntimeError("Loss became NaN/Inf. Inspect batch / masks / logits.")

#         loss_to_backprop = loss / GRAD_ACCUM

#         if scaler.is_enabled():
#             scaler.scale(loss_to_backprop).backward()
#         else:
#             loss_to_backprop.backward()

#         total_loss += float(loss.detach().float().cpu())

#         if (step % GRAD_ACCUM) == 0:
#             if scaler.is_enabled():
#                 scaler.unscale_(optimizer)
#             torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

#             if scaler.is_enabled():
#                 scaler.step(optimizer)
#                 scaler.update()
#             else:
#                 optimizer.step()

#             scheduler.step()
#             optimizer.zero_grad(set_to_none=True)

#     # FLUSH remainder grads if len(train_loader) not divisible by GRAD_ACCUM
#     if last_step != 0 and (last_step % GRAD_ACCUM) != 0:
#         if scaler.is_enabled():
#             scaler.unscale_(optimizer)
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

#         if scaler.is_enabled():
#             scaler.step(optimizer)
#             scaler.update()
#         else:
#             optimizer.step()

#         scheduler.step()
#         optimizer.zero_grad(set_to_none=True)

#     # In your epoch loop, use FAST full eval + LIMITED token metrics
#     # (token metrics are computed on a few batches to keep runtime sane)
#     # ...inside epoch loop, after training...
#     train_stats = stats_on_loader(
#         train_eval_loader,
#         model,
#         DEVICE,
#         USE_AMP,
#         AMP_DTYPE,
#         threshold=THRESH,
#         compute_token_loss=True,
#         compute_token_metrics=True,
#         limit_batches=20,  # <- tune (e.g., 10-50). This is the speed knob.
#     )
#     dev_stats = stats_on_loader(
#         dev_eval_loader,
#         model,
#         DEVICE,
#         USE_AMP,
#         AMP_DTYPE,
#         threshold=THRESH,
#         compute_token_loss=True ,
#         compute_token_metrics=True,
#         limit_batches=None,  # full dev paragraph metrics; token metrics still computed but you can also cap it
#     )

#     print(
#         f"Epoch {epoch} | "
#         f"train loss={train_stats['loss']:.4f} (par={train_stats['loss_par']:.4f}, tok={train_stats['loss_tok']}) | "
#         f"train f1={train_stats['f1']:.4f} acc={train_stats['acc']:.4f} acc0={train_stats['acc_nonpcl']:.4f} acc1={train_stats['acc_pcl']:.4f} | "
#         f"train tok_acc={train_stats.get('tok_acc')} tok_acc0={train_stats.get('tok_acc0')} tok_acc1={train_stats.get('tok_acc1')} | "
#         f"dev loss={dev_stats['loss']:.4f} (par={dev_stats['loss_par']:.4f}, tok={dev_stats['loss_tok']}) | "
#         f"dev f1={dev_stats['f1']:.4f} acc={dev_stats['acc']:.4f} acc0={dev_stats['acc_nonpcl']:.4f} acc1={dev_stats['acc_pcl']:.4f} | "
#         f"dev tok_acc={dev_stats.get('tok_acc')} tok_acc0={dev_stats.get('tok_acc0')} tok_acc1={dev_stats.get('tok_acc1')} | "
#         f"lambda_token={model.lambda_token} | backbone={MODEL_KEY} | amp={USE_AMP} dtype={AMP_DTYPE}"
#     )

In [9]:
# --- Optuna: setup (NEW CELL) ---
import optuna
from optuna.pruners import MedianPruner
import random, gc

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

OPTUNA_DIR = ROOT / "runs" / "optuna_stage05_windowpool"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

storage_url = f"sqlite:///{(OPTUNA_DIR / 'study.db').as_posix()}"
study = optuna.create_study(
    study_name="stage05_windowpool" ,
    direction="maximize",
    storage=storage_url,
    load_if_exists=True,
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=1, interval_steps=1),
)

print("storage:", storage_url)
print("trials so far:", len(study.trials))

[I 2026-03-02 07:08:03,790] Using an existing study with name 'stage05_windowpool' instead of creating a new one.


storage: sqlite:////home/joshua_killa/doc/y3/PCL-detection/runs/optuna_stage05_windowpool/study.db
trials so far: 3


In [10]:
# --- Optuna: objective (NEW CELL) ---
from optuna import trial
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim import AdamW

# fixed knobs (match your notebook)
EPOCHS = 7
BATCH_SIZE = 16
GRAD_ACCUM = 2
THRESH = 0.5
TOKEN_ONLY_EPOCHS = 1
EARLY_STOP_PATIENCE = 4

# reuse the same balancing weights you already computed from train_df
_labels = train_df["label_bin"].astype(int).to_numpy()
_pos = int(_labels.sum())
_neg = int(len(_labels) - _pos)
_w_pos = 1.0 / max(_pos, 1)
_w_neg = 1.0 / max(_neg, 1)
_sample_weights = torch.tensor([_w_pos if y == 1 else _w_neg for y in _labels], dtype=torch.double)

def objective(trial: optuna.Trial):
    set_seed(SEED)

    # requested search params
    tau = trial.suggest_float("tau", 0.20, 0.60)
    alpha_window = trial.suggest_float("alpha_window", 0.2, 3.0, log=True)
    stride = trial.suggest_categorical("stride", [1, 2])

    # extra (useful) params
    lr = trial.suggest_float("lr", 2e-6, 3e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    trial.set_user_attr("model_name", MODEL_NAME)
    trial.set_user_attr("model_key", MODEL_KEY)
    trial.set_user_attr("max_len", int(MAX_LEN))
    trial.set_user_attr("amp", bool(USE_AMP))
    trial.set_user_attr("amp_dtype", str(AMP_DTYPE) if AMP_DTYPE is not None else None)
    trial.set_user_attr("grad_accum", int(GRAD_ACCUM))
    trial.set_user_attr("batch_size", int(BATCH_SIZE))
    trial.set_user_attr("epochs", int(EPOCHS))

    sampler = WeightedRandomSampler(_sample_weights, num_samples=len(_sample_weights), replacement=True)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
    train_eval_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
    dev_eval_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

    model = TokenCLSModel(
        MODEL_NAME,
        window_sizes=WINDOW_SIZES,
        stride=stride,
        tau=tau,
        alpha_window=alpha_window,
    ).to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_update_steps = (len(train_loader) * EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
    warmup_steps = int(0.07 * total_update_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )

    local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))

    best_f1 = -1.0
    no_improve = 0

    try:
        for epoch in range(EPOCHS):

            model.alpha_window = float(alpha_window)

            model.train()
            optimizer.zero_grad(set_to_none=True)

            last_step = 0
            for step, batch in enumerate(train_loader, start=1):
                last_step = step
                batch = {k: v.to(DEVICE) for k, v in batch.items()}

                if USE_AMP:
                    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                        out = model(**batch)
                        loss = out["loss"]
                else:
                    out = model(**batch)
                    loss = out["loss"]

                loss = loss / GRAD_ACCUM

                if local_scaler.is_enabled():
                    local_scaler.scale(loss).backward()
                else:
                    loss.backward()

                if (step % GRAD_ACCUM) == 0:
                    if local_scaler.is_enabled():
                        local_scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                    if local_scaler.is_enabled():
                        local_scaler.step(optimizer)
                        local_scaler.update()
                    else:
                        optimizer.step()

                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

            # flush remainder
            if last_step and (last_step % GRAD_ACCUM) != 0:
                if local_scaler.is_enabled():
                    local_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                if local_scaler.is_enabled():
                    local_scaler.step(optimizer)
                    local_scaler.update()
                else:
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            # eval: dev f1 is the Optuna score
            dev_stats = stats_on_loader(
                dev_eval_loader,
                model,
                DEVICE,
                USE_AMP,
                AMP_DTYPE,
                threshold=THRESH,
                compute_token_loss=True,
                compute_token_metrics=True,
                limit_batches=None,
            )
            dev_f1 = float(dev_stats["f1"])

            train_stats = stats_on_loader(
                train_eval_loader,
                model,
                DEVICE,
                USE_AMP,
                AMP_DTYPE,
                threshold=THRESH,
                compute_token_loss=True,
                compute_token_metrics=True,
                limit_batches=20,  # quick estimate like your manual loop
            )

            print(
                f"[trial {trial.number}] Epoch {epoch} | "
                f"train loss={train_stats['loss']:.4f} (par={train_stats['loss_par']:.4f}, tok={train_stats['loss_tok']}) | "
                f"train f1={train_stats['f1']:.4f} acc={train_stats['acc']:.4f} | "
                f"train tok_acc={train_stats.get('tok_acc')} | "
                f"dev loss={dev_stats['loss']:.4f} (par={dev_stats['loss_par']:.4f}, tok={dev_stats['loss_tok']}) | "
                f"dev f1={dev_stats['f1']:.4f} acc={dev_stats['acc']:.4f} | "
                f"dev tok_acc={dev_stats.get('tok_acc')}"
            )

            trial.set_user_attr(f"epoch_{epoch}_train", {k: float(v) for k, v in train_stats.items() if isinstance(v, (int, float, np.floating))})
            trial.set_user_attr(f"epoch_{epoch}_dev",   {k: float(v) for k, v in dev_stats.items() if isinstance(v, (int, float, np.floating))})

            # track best epoch too
            if dev_f1 >= best_f1:
                trial.set_user_attr("best_epoch", int(epoch))
                trial.set_user_attr("best_dev_stats", {k: float(v) for k, v in dev_stats.items() if isinstance(v, (int, float, np.floating))})
                trial.set_user_attr("best_train_stats", {k: float(v) for k, v in train_stats.items() if isinstance(v, (int, float, np.floating))})

            trial.report(dev_f1, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            if dev_f1 > best_f1 + 1e-6:
                best_f1 = dev_f1
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= EARLY_STOP_PATIENCE:
                    break

        return float(best_f1)

    except torch.cuda.OutOfMemoryError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise optuna.TrialPruned()
    finally:
        del model, optimizer, scheduler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [11]:
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import numpy as np

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=32)
    model.eval()

    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            all_logits.append(out["paragraph_logit"].detach().float().cpu())
            all_labels.append(batch["paragraph_label"].detach().float().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)

    probs = 1.0 / (1.0 + np.exp(-logits))

    best_f1 = 0.0
    best_thresh = 0.5
    for t in np.linspace(0.1, 0.9, 81):
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds, pos_label=1)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = float(t)

    return best_f1, best_thresh

In [12]:
N_TRIALS = 20
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

print("best value (dev f1):", study.best_value)
print("best params:", study.best_params)

Loading weights: 100%|██████████| 25/25 [00:00<00:00, 482.20it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_440919/3778075589.py:66: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))


[trial 3] Epoch 0 | train loss=0.9364 (par=0.8430, tok=0.33407302796840666) | train f1=0.3992 acc=0.7789 | train tok_acc=0.8724605548107662 | dev loss=0.8973 (par=0.8069, tok=0.32322769995891687) | dev f1=0.4397 acc=0.7980 | dev tok_acc=0.8765211694373926
[trial 3] Epoch 1 | train loss=0.7592 (par=0.6961, tok=0.22601596266031265) | train f1=0.5026 acc=0.8484 | train tok_acc=0.9078452098587192 | dev loss=0.7849 (par=0.7158, tok=0.24721539426933636) | dev f1=0.4894 acc=0.8386 | dev tok_acc=0.8973712226680861
[trial 3] Epoch 2 | train loss=0.5859 (par=0.5488, tok=0.1326256889849901) | train f1=0.7117 acc=0.9367 | train tok_acc=0.950925543982675 | dev loss=0.6791 (par=0.6236, tok=0.19834372468970038) | dev f1=0.5462 acc=0.8945 | dev tok_acc=0.9240111374989763
[trial 3] Epoch 3 | train loss=0.5442 (par=0.5140, tok=0.10801818072795868) | train f1=0.7546 acc=0.9477 | train tok_acc=0.9566618541817057 | dev loss=0.6616 (par=0.6082, tok=0.1911868710409511) | dev f1=0.5380 acc=0.8926 | dev tok_ac

[I 2026-03-02 07:49:27,075] Trial 3 finished with value: 0.5555555555555556 and parameters: {'tau': 0.42555627296787757, 'alpha_window': 0.27943340453866594, 'stride': 1, 'lr': 1.087185382823001e-05, 'weight_decay': 0.07662745778794089}. Best is trial 3 with value: 0.5555555555555556.
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 479.77it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/

[trial 4] Epoch 0 | train loss=1.0237 (par=0.8529, tok=0.3487040176987648) | train f1=0.3775 acc=0.7578 | train tok_acc=0.857314117768382 | dev loss=0.9948 (par=0.8265, tok=0.3437373900052273) | dev f1=0.4201 acc=0.7746 | dev tok_acc=0.8583490295635083
[trial 4] Epoch 1 | train loss=0.9047 (par=0.7711, tok=0.2728965349495411) | train f1=0.4703 acc=0.8258 | train tok_acc=0.881832009899969 | dev loss=0.9433 (par=0.7949, tok=0.30304733099359454) | dev f1=0.4663 acc=0.8185 | dev tok_acc=0.8680288264679388
[trial 4] Epoch 2 | train loss=0.6788 (par=0.6071, tok=0.1464484479278326) | train f1=0.6849 acc=0.9281 | train tok_acc=0.9452150149530782 | dev loss=0.7622 (par=0.6558, tok=0.21742209476051907) | dev f1=0.5344 acc=0.8902 | dev tok_acc=0.9191794283842437
[trial 4] Epoch 3 | train loss=0.6205 (par=0.5609, tok=0.12166856471449136) | train f1=0.7556 acc=0.9484 | train tok_acc=0.9551794369392596 | dev loss=0.7420 (par=0.6363, tok=0.21577474745837125) | dev f1=0.5210 acc=0.8859 | dev tok_acc=0

[I 2026-03-02 08:34:23,898] Trial 4 finished with value: 0.5344129554655871 and parameters: {'tau': 0.24199633123022835, 'alpha_window': 0.4896521092558954, 'stride': 1, 'lr': 6.022829041115573e-06, 'weight_decay': 0.07162977882195729}. Best is trial 3 with value: 0.5555555555555556.
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 475.72it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/t

...

[trial 21] Epoch 0 | train loss=1.0933 (par=0.7392, tok=0.2345854863524437) | train f1=0.4646 acc=0.8109 | train tok_acc=0.8829148190161906 | dev loss=1.1093 (par=0.7371, tok=0.24657106625311304) | dev f1=0.4634 acc=0.8109 | dev tok_acc=0.8800753419048399
[trial 21] Epoch 1 | train loss=0.9632 (par=0.7014, tok=0.17344192378222942) | train f1=0.5677 acc=0.8703 | train tok_acc=0.9139295658451068 | dev loss=1.1062 (par=0.7433, tok=0.24036060363957376) | dev f1=0.4848 acc=0.8376 | dev tok_acc=0.8882073540250593


[I 2026-03-02 17:15:55,361] Trial 21 pruned. 
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 480.54it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_440919/3778075589.py:66: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCAL

[trial 22] Epoch 0 | train loss=0.9103 (par=0.7541, tok=0.3891953930258751) | train f1=0.3588 acc=0.7375 | train tok_acc=0.8561668557285759 | dev loss=0.8906 (par=0.7377, tok=0.3807367357340726) | dev f1=0.3986 acc=0.7593 | dev tok_acc=0.8653427237736467
[trial 22] Epoch 1 | train loss=1.0224 (par=0.8779, tok=0.3601580038666725) | train f1=0.3714 acc=0.7461 | train tok_acc=0.8450680622873054 | dev loss=0.9889 (par=0.8492, tok=0.3480474804386948) | dev f1=0.4139 acc=0.7660 | dev tok_acc=0.8528294161002374


[I 2026-03-02 17:30:14,227] Trial 22 pruned. 


best value (dev f1): 0.5736040609137056
best params: {'tau': 0.28318500188735096, 'alpha_window': 0.9149559117779962, 'stride': 2, 'lr': 1.2023895680305613e-05, 'weight_decay': 0.08829108929430884}


In [13]:
# #empty torch cahce

# torch.cuda.empty_cache()